# Match alarm intervals to new telemetry

This notebook uses the established inclusive start-to-clear interval logic and preserves the reference alarm annotation schema.

In [ ]:
from pathlib import Path

import pandas as pd

ALARM_LOGS_PATH = PROJECT_DIR / "data" / "raw" / "vrm_alarm_logs_clean.csv"
OUTPUT_DIR = PROJECT_DIR / "data" / "processed"

def find_project_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "config" / "sites.csv").exists():
            return candidate
    raise FileNotFoundError(
        "Copy config/sites.example.csv to config/sites.csv before running."
    )


PROJECT_DIR = find_project_root()
SITES_CONFIG = PROJECT_DIR / "config" / "sites.csv"
sites_config = pd.read_csv(SITES_CONFIG)
sites_config["site_id"] = pd.to_numeric(sites_config["site_id"], errors="raise").astype(int)
SELECTED_SITES = dict(zip(sites_config["site_id"], sites_config["site_name"]))

indexed_site_frames = {}
for site_id in SELECTED_SITES:
    path = OUTPUT_DIR / f"site_{site_id}_six_month_vrm.csv"
    frame = pd.read_csv(path)
    frame["timestamp_utc"] = pd.to_datetime(
        frame["timestamp_utc"], utc=True, errors="coerce"
    )
    frame.dropna(subset=["timestamp_utc"], inplace=True)
    frame.set_index("timestamp_utc", inplace=True)
    frame.sort_index(inplace=True)
    indexed_site_frames[site_id] = frame

print("Alarm source:", ALARM_LOGS_PATH)
print("Telemetry sites loaded:", len(indexed_site_frames))


## Merge alarm intervals

The alarm merge follows the reference notebook: each stats timestamp inside an alarm's start-to-clear interval is marked active, overlap counts are accumulated, and important alarm text fields are attached.

In [ ]:
def append_unique(existing, new_value):
    if pd.isna(new_value) or new_value == "":
        return existing
    new_value = str(new_value)
    if pd.isna(existing) or existing == "":
        return new_value
    parts = set(str(existing).split(" | "))
    if new_value in parts:
        return existing
    return f"{existing} | {new_value}"


def merge_alarm_intervals(wide_df, alarm_df):
    out = wide_df.copy()
    out.index = pd.to_datetime(out.index, utc=True, errors="coerce")
    out.sort_index(inplace=True)

    alarm_fields = ["idAlarm", "description", "device", "classification", "duration_seconds"]
    for field in alarm_fields:
        out[f"alarm_{field}"] = pd.NA
    out["alarm_active"] = 0
    out["alarm_overlap_count"] = 0

    if alarm_df.empty or out.empty:
        return out

    alarms = alarm_df.copy()
    alarms["alarm_started_at"] = pd.to_datetime(
        alarms["alarm_started_at"], utc=True, errors="coerce"
    )
    alarms["alarm_cleared_at"] = pd.to_datetime(
        alarms["alarm_cleared_at"], utc=True, errors="coerce"
    )
    alarms["alarm_cleared_at"] = alarms["alarm_cleared_at"].fillna(
        alarms["alarm_started_at"]
    )
    alarms.dropna(subset=["alarm_started_at"], inplace=True)
    alarms = alarms[
        (alarms["alarm_started_at"] <= out.index.max())
        & (alarms["alarm_cleared_at"] >= out.index.min())
    ].sort_values("alarm_started_at")

    for _, alarm in alarms.iterrows():
        mask = (
            (out.index >= alarm["alarm_started_at"])
            & (out.index <= alarm["alarm_cleared_at"])
        )
        if not mask.any():
            continue

        out.loc[mask, "alarm_active"] = 1
        out.loc[mask, "alarm_overlap_count"] += 1
        for field in alarm_fields:
            output_field = f"alarm_{field}"
            out.loc[mask, output_field] = out.loc[mask, output_field].apply(
                lambda existing: append_unique(existing, alarm.get(field))
            )

    return out


vrm_alarm_logs_clean = pd.read_csv(ALARM_LOGS_PATH)
merged_site_frames = {}
alarm_summary_rows = []

for site_id, indexed in indexed_site_frames.items():
    site_alarms = vrm_alarm_logs_clean[
        vrm_alarm_logs_clean["site_id"] == site_id
    ].copy()
    merged = merge_alarm_intervals(indexed, site_alarms)
    merged_site_frames[site_id] = merged
    globals()[f"site_stats_{site_id}_merged"] = merged

    merged.reset_index().to_csv(
        OUTPUT_DIR / f"site_{site_id}_six_month_vrm_with_alarms.csv",
        index=False,
    )
    alarm_summary_rows.append({
        "site_id": site_id,
        "alarm_events_in_log": len(site_alarms),
        "stats_rows_with_active_alarm": int(merged["alarm_active"].sum()),
        "maximum_alarm_overlap": int(merged["alarm_overlap_count"].max()),
    })

alarm_merge_summary = pd.DataFrame(alarm_summary_rows)
alarm_merge_summary.to_csv(OUTPUT_DIR / "selected_sites_alarm_merge_summary.csv", index=False)
alarm_merge_summary

## Alarm mapping verification

In [ ]:
diagnostic_rows = []
for site_id, telemetry in indexed_site_frames.items():
    alarms = vrm_alarm_logs_clean[
        vrm_alarm_logs_clean["site_id"] == site_id
    ].copy()
    alarms["alarm_started_at"] = pd.to_datetime(
        alarms["alarm_started_at"], utc=True, errors="coerce"
    )
    alarms["alarm_cleared_at"] = pd.to_datetime(
        alarms["alarm_cleared_at"], utc=True, errors="coerce"
    )
    alarms["alarm_cleared_at"] = alarms["alarm_cleared_at"].fillna(
        alarms["alarm_started_at"]
    )
    valid = alarms.dropna(subset=["alarm_started_at"]).copy()
    in_range = valid[
        (valid["alarm_started_at"] <= telemetry.index.max())
        & (valid["alarm_cleared_at"] >= telemetry.index.min())
    ]
    mapped = 0
    for alarm in in_range.itertuples(index=False):
        left = telemetry.index.searchsorted(alarm.alarm_started_at, side="left")
        right = telemetry.index.searchsorted(alarm.alarm_cleared_at, side="right")
        mapped += int(right > left)

    merged = merged_site_frames[site_id]
    diagnostic_rows.append({
        "site_id": site_id,
        "telemetry_rows": len(telemetry),
        "telemetry_range_start": telemetry.index.min(),
        "telemetry_range_end": telemetry.index.max(),
        "alarm_rows_total": len(alarms),
        "alarm_rows_with_valid_start": len(valid),
        "alarms_overlapping_telemetry_range": len(in_range),
        "mapped_alarm_events_in_range": mapped,
        "unmapped_alarm_events_in_range": len(in_range) - mapped,
        "unmapped_alarm_events_total": len(alarms) - mapped,
        "alarm_active_telemetry_rows": int(merged["alarm_active"].sum()),
        "maximum_alarm_overlap": int(merged["alarm_overlap_count"].max()),
    })

alarm_matching_verification = pd.DataFrame(diagnostic_rows)
alarm_matching_verification.to_csv(
    OUTPUT_DIR / "alarm_matching_verification.csv", index=False
)
alarm_matching_verification
